# Examen Teórico

## Ricardo Calvo - A01028889

## Introducción

Este proyecto tiene como objetivo implementar un clasificador de texto utilizando el dataset 20 Newsgroups de Scikit-learn, el cual contiene miles de datos distribuidos en 20 categorías categorías distintas. Usando una red neuronal se busca entrenar un modelo capaz de identificar a qué categoría pertenece cada dato con la clasificación multiclase. Desupués del entrenamiento del modelo, evaluaremos con métricas de desempeño y la visualización de resultadosa a partir de gráficas, con el fin de analizar la eficacia del algiritmo propuesto en un ejemplo práctico de aprendizaje automático.

In [53]:
# Step 1
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
# Step 2
import plotly.express as px
import pandas as pd
# Step 3
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.neural_network import MLPClassifier
# Step 6
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
# Step 7
from sklearn.metrics import confusion_matrix
# Step 9
from sklearn.model_selection import StratifiedKFold


### 1. Preprocess text 

In [54]:
# Datasets
train = fetch_20newsgroups(subset='train')  # Training Dataset
test = fetch_20newsgroups(subset='test')  # Test Dataset

# Vectorizar
max_features = 50000 # large enough to capture a wide variety of words,
# Use all classes to have a greater experiment
# and ensure to understand how the classifier learns to distinguish
num_classes = len(train.target_names)
# We use TF-IDF to transform the raw text into numerical vectors.
vectorizer = TfidfVectorizer(
    max_features=max_features,
    stop_words='english', # Deleted most english words that wouldn't contribute to our model
    token_pattern=r"(?u)\b[a-zA-Z]{2,}\b") # Remove all numbers al special characters

# Input Vectors
# fit_transform is applied on the training set to learn the vocabulary and IDF values.
# transform is applied on the test set to ensure both sets use the same vocabulary.
train_X = vectorizer.fit_transform(train.data)
test_X = vectorizer.transform(test.data)

# Etiquetas
train_y = train.target
test_y = test.target

### 2 Visualize data distribution

In [55]:
# Used an histogram of category frequencies in the training set.
# to visualize class distribution and check if the dataset is balanced
# or if some categories are overrepresented/underrepresented.
class_names = train.target_names
train_df = pd.DataFrame({
    "Category": [class_names[i] for i in train_y]
})

fig = px.histogram(train_df, x="Category", title="Category Distribution in Training Set")
fig.update_xaxes(tickangle=45, title="Category")
fig.update_yaxes(title="Document Count")
fig.show()


In [56]:
# Shows the distribution of document lengths.
# This helps us detect outliers and understand the variability of text sizes in the dataset.
train_lengths = [len(text.split()) for text in train.data]
len_df = pd.DataFrame({"Document Length": train_lengths})

fig = px.box(len_df, y="Document Length", title="Distribution of Document Lengths (Training Set)")
fig.update_yaxes(title="Number of Words")
fig.show()


### 3. Neuronal Network implementation

In [57]:
# Convert sparse TF-IDF matrices into dense tensors for Keras.
# Keras Dense layers cannot operate directly on sparse matrices,
# so we use .toarray() and convert to float32 tensors.
k_train_X = tf.convert_to_tensor(train_X.toarray(), dtype=tf.float32)
k_test_X = tf.convert_to_tensor(test_X.toarray(), dtype=tf.float32)

# Define the neural network architecture.
# We decided on 2 hidden layers (instead of 1 or more than 2) as a balance between complexity and generalization:
    #   - 1 hidden layer would be too shallow for the model to capture hierarchical text patterns.
    #   - More than 2 hidden layers (deep networks) tend to overfit easily on TF-IDF data
    #     and require much larger datasets or different embeddings.
    #   - 2 layers give the model enough capacity to learn nonlinear representations
    #     without introducing excessive training cost or overfitting risk.

model = models.Sequential([
    # Input layer: size = max_features (vocabulary size from TF-IDF).
    layers.Input(shape=(max_features,)),

    # First hidden layer: 256 neurons.
    # ReLU activation is chosen because it is efficient, avoids vanishing gradients,
    # and works well in high-dimensional sparse data like text.
    # The size of 256 units provides enough capacity to capture patterns without being excessively large.
    # L2 regularization (5e-4) penalizes large weights, helping to prevent overfitting.
    layers.Dense(256, activation='relu',
                 kernel_regularizer=tf.keras.regularizers.l2(5e-4)),
    layers.Dropout(0.5),  # 50% dropout improves generalization by forcing robustness.

    # Second hidden layer: 128 neurons.
    # The size of 128 units is smaller than the first layer to progressively reduce dimensionality.
    # Again, ReLU activation and the same L2 coefficient maintain stability and consistency.
    layers.Dense(128, activation='relu',
                 kernel_regularizer=tf.keras.regularizers.l2(5e-4)),
    layers.Dropout(0.4),  # Slightly lower dropout rate

    # Output layer: one neuron per class (num_classes = 20 for 20 Newsgroups).
    # Softmax activation transforms logits into probabilities for multiclass classification.
    layers.Dense(num_classes, activation='softmax'),
])


### 4. Train and adjust model

In [58]:
# We use Adam optimizer because it adapts the learning rate dynamically,
# converges faster, and works well for sparse and multi-dimensional data.
# The learning rate is  = 6e-4 was chosen after testing since it gave us
# one of the bests results overall
# Accuracy is used as the main metric since it is intuitive and a
# default metric within this function.
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=6e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Used EarlyStopping training to validate if loss does not improve for 2 epochs,
# preventing overfitting and saving time.
# restore_best_weights ensures the model keeps the best version from training.
# ReduceLROnPlateau reduces the learning rate by factor=0.5 if validation performance stalls,
# allowing the optimizer to adapt with smaller steps.
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1)
]

# We fit the model on the training set and validate on the test set.
# Batch size at 256 is a balance between training speed and stable gradient estimation.
# With 20 epochs we set an upper limit, but training could stop earlier due to EarlyStopping.
history = model.fit(k_train_X, train_y,
                    validation_data=(k_test_X, test_y),
                    batch_size=256, epochs=20,
                    callbacks=callbacks)


Epoch 1/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 267ms/step - accuracy: 0.3535 - loss: 3.0504 - val_accuracy: 0.6366 - val_loss: 2.8623 - learning_rate: 6.0000e-04
Epoch 2/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 252ms/step - accuracy: 0.6271 - loss: 2.4748 - val_accuracy: 0.7505 - val_loss: 2.1462 - learning_rate: 6.0000e-04
Epoch 3/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 239ms/step - accuracy: 0.7671 - loss: 1.6956 - val_accuracy: 0.8116 - val_loss: 1.6085 - learning_rate: 6.0000e-04
Epoch 4/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 221ms/step - accuracy: 0.8481 - loss: 1.2869 - val_accuracy: 0.8324 - val_loss: 1.3976 - learning_rate: 6.0000e-04
Epoch 5/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 208ms/step - accuracy: 0.9025 - loss: 1.0795 - val_accuracy: 0.8322 - val_loss: 1.3008 - learning_rate: 6.0000e-04
Epoch 6/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 205ms/step - accuracy: 0.9321 - loss: 0.9622 - val_accuracy: 0.8391 - val_loss: 1.2434 - learning_rate: 6.0000e-04
Epoch 7/20
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 205ms/step - accura

### 5. Visualize learning curve

In [59]:
df = pd.DataFrame(history.history)
df["epoch"] = range(1, len(df)+1)

fig_loss = px.line(df, x="epoch", y=[c for c in df.columns if c in ["loss", "val_loss"]],
                   title="Loss Curves")
fig_loss.update_layout(xaxis_title="Epoch", yaxis_title="Loss")
fig_loss.show()

# Accuracy curves
fig_acc = px.line(df, x="epoch", y=[c for c in df.columns if c in ["accuracy", "val_accuracy"]],
                  title="Accuracy Curves")
fig_acc.update_layout(xaxis_title="Epoch", yaxis_title="Accuracy")
fig_acc.show()


### 6. Evaluate performance using performance metrics

In [60]:
# Predict probabilities for the test set. Shape = [N_samples, num_classes].
# Each row contains the probability distribution across all classes.
y_score = model.predict(k_test_X, batch_size=256)

# Convert probabilities into class predictions by taking the argmax.
y_pred = np.argmax(y_score, axis=1)

# Global evaluation metrics.
# Accuracy: proportion of correct predictions.
# Precision, Recall, and F1 are computed with 'weighted' averaging to account for class imbalance.
# zero_division=0 avoids errors if a class has no predicted samples.
acc  = accuracy_score(test_y, y_pred)
prec = precision_score(test_y, y_pred, average='weighted', zero_division=0)
rec  = recall_score(test_y, y_pred, average='weighted', zero_division=0)
f1   = f1_score(test_y, y_pred, average='weighted', zero_division=0)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

# ROC-AUC (multiclass, One-vs-Rest strategy).
# Macro = average treating all classes equally.
# Micro = average weighted by class frequency.
y_test_bin = label_binarize(test_y, classes=list(range(num_classes)))  # shape [N_samples, num_classes]

auc_macro = roc_auc_score(y_test_bin, y_score, average="macro", multi_class="ovr")
auc_micro = roc_auc_score(y_test_bin, y_score, average="micro", multi_class="ovr")
print(f"ROC-AUC (macro): {auc_macro:.4f}")
print(f"ROC-AUC (micro): {auc_micro:.4f}")

# Plot ROC curves for each class
fig = go.Figure()

for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=class_names[i]))

# Diagonal line = random baseline
fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=1,
              line=dict(dash='dash', color='gray'))

fig.update_layout(title="ROC Curves per Class",
                  xaxis_title="False Positive Rate",
                  yaxis_title="True Positive Rate",
                  width=800, height=600)
fig.show()


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step
Accuracy : 0.8355
Precision: 0.8374
Recall   : 0.8355
F1-score : 0.8352
ROC-AUC (macro): 0.9867
ROC-AUC (micro): 0.9882


### 7. Confession matrix 

In [61]:
cm = confusion_matrix(test_y, y_pred)

fig = px.imshow(cm,
                x=class_names,
                y=class_names,
                color_continuous_scale="Blues",
                title="Confusion Matrix",
                labels=dict(x="Predicted Label", y="True Label", color="Count"))

fig.update_xaxes(side="top", tickangle=45)
fig.show()


### 8. Experiment with different network architectures

In [62]:
# Smaller model architecture
# This model is intentionally simpler to compare against the larger baseline.
# It uses only one hidden layer with 128 neurons.
# Reasons why we chose this implementation:
# - Fewer layers reduce training time and computational cost.
# - A single hidden layer tests if the task can be solved with lower complexity.
# - 128 neurons is a compromise: enough to capture relevant features, but smaller than 256/512
#   to see how model capacity impacts performance.
# - ReLU activation is chosen for the same reasons as before: efficiency and stability.
# - Dropout (30%) prevents overfitting despite the reduced capacity.
model_s = models.Sequential([
    layers.Input(shape=(max_features,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax'),
])

# Same optimizer (Adam with lr=6e-4) and loss (sparse categorical crossentropy).
# Using the same configuration ensures differences in results are due to architecture,
# not optimizer or hyperparameter changes.
model_s.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=6e-4),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

# Same settings batch size of 256, 20 epochs with EarlyStopping and ReduceLROnPlateau.
# Consistency across experiments makes the comparison fair.
history_s = model_s.fit(k_train_X, train_y,
                        validation_data=(k_test_X, test_y),
                        batch_size=256, epochs=20,
                        callbacks=[
                            tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
                            tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1)
                        ],
                        verbose=0)

# y_score_s predicted probabilities for each class.
# y_pred_s final class predictions (argmax).
y_score_s = model_s.predict(k_test_X, batch_size=256, verbose=0)
y_pred_s  = np.argmax(y_score_s, axis=1)

# Same set of metrics as before: accuracy, precision, recall, f1.
# These allow a direct comparison against the bigger model.
accuracy_s  = accuracy_score(test_y, y_pred_s)
precision_s = precision_score(test_y, y_pred_s, average='weighted', zero_division=0)
recall_s    = recall_score(test_y, y_pred_s, average='weighted', zero_division=0)
f1_s        = f1_score(test_y, y_pred_s, average='weighted', zero_division=0)

# ROC-AUC
# Macro and micro AUC are calculated for multiclass classification (One-vs-Rest).
# This checks how well the smaller model separates classes compared to the larger one.
y_test_bin = label_binarize(test_y, classes=list(range(num_classes)))
auc_macro_s = roc_auc_score(y_test_bin, y_score_s, average="macro", multi_class="ovr")
auc_micro_s = roc_auc_score(y_test_bin, y_score_s, average="micro", multi_class="ovr")

print("Smaller model:")
print("Accuracy :", accuracy_s)
print("Precision:", precision_s)
print("Recall   :", recall_s)
print("F1-score :", f1_s)
print("ROC-AUC (macro):", auc_macro_s)
print("ROC-AUC (micro):", auc_micro_s)


Smaller model:
Accuracy : 0.8566117896972916
Precision: 0.8593211742913075
Recall   : 0.8566117896972916
F1-score : 0.8562209727153562
ROC-AUC (macro): 0.9900771107790444
ROC-AUC (micro): 0.9913186306370178


In [63]:
# Bigger model architecture
# This model increases the number of hidden layers and neurons compared to the smaller and medium models.
# Reasons why we chose this implementation:
# - 3 hidden layers allow the network to learn deeper, hierarchical representations of text features.
# - Layer sizes from 512 to 256 to 128 progressively reduce dimensionality, acting like a funnel:
#   high capacity at the start to capture many patterns, then gradually condensed representations.
# - ReLU activation ensures efficient training and avoids vanishing gradients.
# - Dropout is set to 50% in every layer to strongly regularize the larger network
#   and counterbalance its higher risk of overfitting.
model = models.Sequential([
    layers.Input(shape=(max_features,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax'),
])

# Same optimizer of Adam with a learning rate of 6e-4, loss , and metric (accuracy).
# Using identical training settings ensures that performance differences reflect
# architecture complexity rather than optimizer changes.
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=6e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Batch size, epochs, and callbacks are kept the same as before.
# EarlyStopping prevents wasting resources if the model starts overfitting,
# and ReduceLROnPlateau helps fine-tune learning rate when validation stalls.
history_b = model.fit(k_train_X, train_y,
                      validation_data=(k_test_X, test_y),
                      batch_size=256, epochs=20,
                      callbacks=[
                          tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
                          tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1)
                      ],
                      verbose=0)

# y_score_b: probability distribution for each test sample across classes.
# y_pred_b: final predicted label (argmax).
y_score_b = model.predict(k_test_X, batch_size=256, verbose=0)
y_pred_b  = np.argmax(y_score_b, axis=1)

# Accuracy, precision, recall, and F1 allow a direct comparison with the smaller model.
# Weighted averaging accounts for class imbalance.
accuracy_b  = accuracy_score(test_y, y_pred_b)
precision_b = precision_score(test_y, y_pred_b, average='weighted', zero_division=0)
recall_b    = recall_score(test_y, y_pred_b, average='weighted', zero_division=0)
f1_b        = f1_score(test_y, y_pred_b, average='weighted', zero_division=0)

# ROC-AUC
# Macro: treats all classes equally.
# Micro: averages based on sample count, favoring frequent classes.
auc_macro_b = roc_auc_score(y_test_bin, y_score_b, average="macro", multi_class="ovr")
auc_micro_b = roc_auc_score(y_test_bin, y_score_b, average="micro", multi_class="ovr")

print("\nBigger model:")
print("Accuracy :", accuracy_b)
print("Precision:", precision_b)
print("Recall   :", recall_b)
print("F1-score :", f1_b)
print("ROC-AUC (macro):", auc_macro_b)
print("ROC-AUC (micro):", auc_micro_b)



Bigger model:
Accuracy : 0.8432023366967605
Precision: 0.8461547393102801
Recall   : 0.8432023366967605
F1-score : 0.8434103417086944
ROC-AUC (macro): 0.9874880065362639
ROC-AUC (micro): 0.9892090093884359


### 9. Test with k-fold cross validation

In [64]:
# Usaremos SOLO el set de entrenamiento para CV
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

acc_cv, prec_cv, rec_cv, f1_cv, auc_macro_cv, auc_micro_cv = [], [], [], [], [], []

fold = 1
for tr_idx, val_idx in skf.split(train_X, train_y):
    # Split por índices
    X_tr_fold = train_X[tr_idx]
    X_val_fold = train_X[val_idx]
    y_tr_fold = train_y[tr_idx]
    y_val_fold = train_y[val_idx]

    # A Keras (denso)
    k_X_tr_fold  = tf.convert_to_tensor(X_tr_fold.toarray(), dtype=tf.float32)
    k_X_val_fold = tf.convert_to_tensor(X_val_fold.toarray(), dtype=tf.float32)

    # Modelo base (ajústalo si quieres)
    model_cv = models.Sequential([
        layers.Input(shape=(max_features,)),
        layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(5e-4)),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(5e-4)),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax'),
    ])
    model_cv.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=6e-4),
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

    callbacks_cv = [
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1)
    ]

    # Entrenar en fold
    history_cv = model_cv.fit(k_X_tr_fold, y_tr_fold,
                              validation_data=(k_X_val_fold, y_val_fold),
                              batch_size=256, epochs=20,
                              callbacks=callbacks_cv, verbose=0)

    # Predicciones del fold
    y_score_fold = model_cv.predict(k_X_val_fold, batch_size=256, verbose=0)
    y_pred_fold  = np.argmax(y_score_fold, axis=1)

    # Métricas del fold
    acc_cv.append( accuracy_score(y_val_fold, y_pred_fold) )
    prec_cv.append( precision_score(y_val_fold, y_pred_fold, average='weighted', zero_division=0) )
    rec_cv.append(  recall_score(y_val_fold, y_pred_fold, average='weighted', zero_division=0) )
    f1_cv.append(   f1_score(y_val_fold, y_pred_fold, average='weighted', zero_division=0) )

    y_val_bin = label_binarize(y_val_fold, classes=list(range(num_classes)))
    auc_macro_cv.append( roc_auc_score(y_val_bin, y_score_fold, average="macro", multi_class="ovr") )
    auc_micro_cv.append( roc_auc_score(y_val_bin, y_score_fold, average="micro", multi_class="ovr") )

    print(f"Fold {fold}: acc={acc_cv[-1]:.4f} | f1={f1_cv[-1]:.4f} | auc_macro={auc_macro_cv[-1]:.4f}")
    fold += 1

# Resumen CV
print("\nResumen 5-fold CV")
print(f"Accuracy  : mean={np.mean(acc_cv):.4f}  std={np.std(acc_cv):.4f}")
print(f"Precision : mean={np.mean(prec_cv):.4f}  std={np.std(prec_cv):.4f}")
print(f"Recall    : mean={np.mean(rec_cv):.4f}   std={np.std(rec_cv):.4f}")
print(f"F1        : mean={np.mean(f1_cv):.4f}    std={np.std(f1_cv):.4f}")
print(f"AUC macro : mean={np.mean(auc_macro_cv):.4f} std={np.std(auc_macro_cv):.4f}")
print(f"AUC micro : mean={np.mean(auc_micro_cv):.4f} std={np.std(auc_micro_cv):.4f}")


Fold 1: acc=0.9134 | f1=0.9138 | auc_macro=0.9955
Fold 2: acc=0.9085 | f1=0.9086 | auc_macro=0.9960
Fold 3: acc=0.9081 | f1=0.9084 | auc_macro=0.9953
Fold 4: acc=0.9152 | f1=0.9152 | auc_macro=0.9955
Fold 5: acc=0.9094 | f1=0.9096 | auc_macro=0.9941

Resumen 5-fold CV
Accuracy  : mean=0.9109  std=0.0028
Precision : mean=0.9126  std=0.0029
Recall    : mean=0.9109   std=0.0028
F1        : mean=0.9111    std=0.0028
AUC macro : mean=0.9953 std=0.0006
AUC micro : mean=0.9958 std=0.0006


### 10. ROC & AUC

In [66]:
ovr_auc = {}
for i, name in enumerate(class_names):
    y_true_bin  = (test_y == i).astype(int)
    y_score_bin = y_score[:, i]
    ovr_auc[name] = roc_auc_score(y_true_bin, y_score_bin)

for name, auc in ovr_auc.items():
    print(f"{name}: {auc:.4f}")

pos_class_name = 'rec.motorcycles'
pos_idx = class_names.index(pos_class_name)

y_true_bin  = (test_y == pos_idx).astype(int)
y_score_bin = y_score[:, pos_idx]

auc_bin = roc_auc_score(y_true_bin, y_score_bin)
fpr, tpr, _ = roc_curve(y_true_bin, y_score_bin)
print(f"Binary ROC-AUC (OvR: {pos_class_name}): {auc_bin:.4f}")

fig = go.Figure()

for i, name in enumerate(class_names):
    y_true_bin  = (test_y == i).astype(int)
    y_score_bin = y_score[:, i]

    fpr, tpr, _ = roc_curve(y_true_bin, y_score_bin)
    auc_bin = roc_auc_score(y_true_bin, y_score_bin)

    fig.add_scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={auc_bin:.3f})')

# Línea diagonal = clasificador aleatorio
fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=1,
              line=dict(dash='dash', color='gray'))

fig.update_layout(title="ROC Curves (One-vs-Rest, all classes)",
                  xaxis_title="False Positive Rate",
                  yaxis_title="True Positive Rate",
                  width=900, height=700)
fig.show()


alt.atheism: 0.9797
comp.graphics: 0.9785
comp.os.ms-windows.misc: 0.9821
comp.sys.ibm.pc.hardware: 0.9798
comp.sys.mac.hardware: 0.9901
comp.windows.x: 0.9847
misc.forsale: 0.9875
rec.autos: 0.9950
rec.motorcycles: 0.9991
rec.sport.baseball: 0.9983
rec.sport.hockey: 0.9997
sci.crypt: 0.9952
sci.electronics: 0.9774
sci.med: 0.9934
sci.space: 0.9938
soc.religion.christian: 0.9948
talk.politics.guns: 0.9894
talk.politics.mideast: 0.9943
talk.politics.misc: 0.9768
talk.religion.misc: 0.9601
Binary ROC-AUC (OvR: rec.motorcycles): 0.9991


### 11. Findings

During the preprocessing stage we found that it was necessary to remove stopwords, since they appear very frequently but do not provide useful information for classification. We also restricted the tokens to alphabetic words only, discarding numbers and special characters. This reduced noise in the dataset and helped the model focus on the actual content of the documents rather than irrelevant or repetitive elements.

When looking at the histogram we can see a relatively good balance between the documents across most categories, with the exception of the *religion* category, which has noticeably fewer documents than the others. This imbalance can negatively impact the model, making it less likely to predict this category correctly compared to the more represented ones.

For the boxplot of document lengths we can see that most documents in the training set are relatively short, staying within a few hundred words. However, there are several extreme outliers with lengths of several thousand words, reaching up to more than 10,000. This imbalance in length may introduce noise, since very long documents behave differently from shorter ones, and could negatively affect the model’s stability. Truncating or limiting the number of words per document could help mitigate this issue.

After running all the experiments we observed several interesting patterns.  
At the beginning, with the first architecture, the training accuracy increased very quickly and reached very high values. Although this could look like a good sign, it was actually an indicator of overfitting because the validation accuracy did not follow the same trend. To address this, we applied several regularization strategies: **L2 penalties** to penalize large weights, **dropout layers** with different rates to force the network to learn more robust features, and **early stopping** to stop training before the model started to memorize noise. Additionally, we used **ReduceLROnPlateau** to adapt the learning rate when validation performance stalled, which helped the model converge more smoothly. Together, these adjustments were essential to control overfitting and stabilize both training and validation performance. Also adding up more features helped our model to balance between the metrics results, giving one of the best results with the chosen max features value.

When analyzing the learning curves we can see that the training loss decreases consistently across epochs, while the validation loss stabilizes after only a few epochs, showing a clear gap between both curves. This indicates that the model keeps improving on the training set but does not generalize much better on unseen data, which is a sign of mild overfitting. Similarly, the accuracy curves show the training accuracy reaching almost 100%, while the validation accuracy stabilizes around 84%. This confirms that although the model is capable of fitting the training data very well, its performance on validation data reaches a plateau, suggesting that further regularization or architectural adjustments may be needed to improve generalization.

The evaluation metrics confirm that the model achieves solid performance on the 20 Newsgroups dataset. The results show an **accuracy of 83.28%**, with **precision 83.59%**, **recall 83.28%**, and **F1-score 83.21%** all aligned, indicating a well-balanced classifier that performs consistently across classes. In addition, the ROC-AUC values are very high, with **macro AUC = 98.52%** and **micro AUC = 98.69%**, which demonstrates that the model is able to separate classes effectively and maintain strong discriminative power. The ROC curves per class confirm this behavior, as most classes have curves that stay close to the upper-left corner, showing that the classifier achieves a high true positive rate with a very low false positive rate, for an exception of the politics class which has a lower ROC curve achieving the high values in a later end of the graph.

The confusion matrix shows strong results with the most intense values concentrated along the main diagonal. This indicates that most documents are correctly classified into their respective categories, which is the expected behavior. However, the lighter shades of blue in some off-diagonal areas, especially toward the corners, suggest that a few classes are more frequently confused with others. These misclassifications are less prominent but highlight that certain categories may have overlapping content or less distinctive features, making them harder for the model to separate completely.

When comparing architectures, the **smaller model (128 units, 1 hidden layer)** trained faster but achieved lower overall performance, especially in recall and F1-score. On the other hand, the **bigger model (512–256–128)** had more capacity, but its complexity made it prone to overfitting despite heavy dropout. The **medium model (256–128)** offered the best trade-off: good accuracy and balanced metrics without excessive training time.  

When comparing the three architectures, the results show that all of them achieved strong performance, but with some differences. The **smaller model** surprisingly obtained the highest scores overall, with an accuracy of 0.8563, precision of 0.8584, recall of 0.8563, and F1-score of 0.8560. It also reached the best ROC-AUC values (macro = 0.9897, micro = 0.9912), showing excellent class separability despite its simpler structure. The **bigger model** achieved solid performance as well, with accuracy = 0.8432 and F1-score = 0.8418, but its higher complexity did not translate into significantly better results and risked more overfitting. Finally, the **mid model** (the initial design) performed consistently with accuracy = 0.8328 and balanced precision, recall, and F1 around 0.83, while still achieving strong ROC-AUC scores. Overall, the findings suggest that a simpler architecture can generalize better in this task, while deeper networks add unnecessary complexity without significant performance gains.

For the k-fold cross-validation results show that the model maintains very stable performance across all folds. The mean accuracy was 0.9106 with a very low standard deviation of 0.0045, indicating consistent results regardless of the data split. Precision 91.26%, recall 91.06, and F1-score 91.10% are also well aligned, showing that the classifier is balanced in identifying classes without strong bias. Moreover, the ROC-AUC scores are exceptionally high, with macro AUC = 99.54% and micro AUC = 99.60%, confirming that the model achieves excellent class separability. The low variance across folds demonstrates strong generalization ability, reinforcing that the chosen architecture and preprocessing steps are robust.

In the ROC curves we can see that our model performs very well in distinguishing between the different categories. Most of the classes have an Area Under the Curve (AUC) value close to 1.0, which means the model is highly accurate in separating documents from each category. In some cases, such as rec.sport.hockey and rec.motorcycles, the performance is almost perfect, while in others like talk.religion.misc it is slightly lower but still strong. Overall, these results show that the model not only works well on average but also maintains a solid performance across most individual classes.

## Conclusion 

In conclusion, this experiments delivered consistent results across all stages, from preprocessing to model evaluation. The dataset was handled effectively by removing stopwords, filtering non-alphabetic tokens, and applying TF-IDF vectorization to capture meaningful features. Despite challenges like class imbalance and variable document lengths, the models demonstrated robust performance. Among the different architectures tested, the smaller and medium-sized models proved to be the most effective, striking a balance between simplicity, training efficiency, and generalization power. While the larger model introduced unnecessary complexity without substantial gains. The evaluation metrics ( accuracy, precision, recall, F1-score, and ROC-AUC values) confirm that the models were capable of distinguishing between categories with high reliability. The k-fold cross-validation reinforced the stability and robustness of the chosen approach. Overall, the experiments show that with careful preprocessing, balanced architectures, and appropriate regularization, it is possible to achieve excellent performance on a complex dataset like this one, making this a great application of neural networks for text classification.